# Delete Some Parent labels with all child labels

In [5]:
import json

# Load your JSON data
with open('category_tree_with_counts.json', 'r', encoding='utf-8') as f:
    data = json.load(f)


# List of parent categories to delete
categories_to_delete = [
    "00- نرم افزارهای کاربردی",
    "000- کتاب های صوتی",
    "28- نسخ خطی",
    "27- کتاب های تصویری",
    "24- مجلات، مقالات و نشریات",
    "23- پرسش و پاسخ",
    "25- کتب مرجع",
    
]

# Function to delete categories recursively
def delete_categories(data, categories_to_delete):
    if isinstance(data, dict):
        # Remove the specified categories
        for category in categories_to_delete:
            if category in data:
                del data[category]
        
        # Recursively process subcategories
        for key in list(data.keys()):
            if isinstance(data[key], dict):
                if 'subcategories' in data[key]:
                    delete_categories(data[key]['subcategories'], categories_to_delete)
                # Also check if the value itself has subcategories structure
                delete_categories(data[key], categories_to_delete)

# Delete the specified categories
delete_categories(data, categories_to_delete)

# Save the modified data
with open('category_tree_modified.json', 'w', encoding='utf-8') as f:
    json.dump(data, f, ensure_ascii=False, indent=4)

print("Categories deleted successfully!")

Categories deleted successfully!


# Count the number of leaf tags

In [2]:
import json

def quick_leaf_count(input_file):
    with open(input_file, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    def count(node):
        if not node: return 0
        total = 0
        for key, value in node.items():
            if key in ['book_count', 'url', 'subcategories']: continue
            subcats = value.get('subcategories', {})
            has_children = any(k not in ['book_count', 'url', 'subcategories'] for k in subcats)
            total += (0 if has_children else 1) + count(subcats)
        return total
    
    return count(data)

# Usage
print(f"Leaf tags: {quick_leaf_count('category_tree_modified.json')}")

Leaf tags: 799


# finding single level categories

In [10]:
import json

def find_root_single_level_categories(data):
    """
    Find only root-level categories that have no subcategories at all
    """
    root_single_level = []
    
    for category_name, category_data in data.items():
        # Skip metadata fields
        if category_name in ['book_count', 'url', 'subcategories']:
            continue
            
        # Check if this root category has any actual subcategories
        has_actual_subcategories = False
        
        if "subcategories" in category_data:
            subcats = category_data["subcategories"]
            if subcats and isinstance(subcats, dict):
                # Look for actual category names (not metadata)
                for key in subcats.keys():
                    if key not in ['book_count', 'url', 'subcategories']:
                        has_actual_subcategories = True
                        break
        
        # If no actual subcategories found, this is a root single-level category
        if not has_actual_subcategories:
            root_single_level.append({
                'name': category_name,
                'book_count': category_data.get('book_count', 0),
                'url': category_data.get('url', '')
            })
    
    return root_single_level

def main():
    # Load your JSON data
    with open('category_tree_modified.json', 'r', encoding='utf-8') as file:
        data = json.load(file)
    
    # Find root single-level categories
    root_single_cats = find_root_single_level_categories(data)
    
    print(f"Found {len(root_single_cats)} root-level single-level categories:\n")
    
    # Display results
    for i, category in enumerate(root_single_cats, 1):
        print(f"{i:2d}. {category['name']}")
        print(f"    Books: {category['book_count']}")
        if category['url']:
            print(f"    URL: {category['url']}")
        print()

if __name__ == "__main__":
    main()

Found 2 root-level single-level categories:

 1. 05- شخصیت ها، اصحاب و علماء
    Books: 1024
    URL: https://www.ghbook.ir/index.php?option=com_dbook&task=listcats&cat_id=4858&Itemid=&lang=fa

 2. 22- حجاب و عفاف
    Books: 79
    URL: https://www.ghbook.ir/index.php?option=com_dbook&task=listcats&cat_id=586&Itemid=&lang=fa



# finding all leaf tags with few books

In [12]:
import json

def analyze_leaf_categories_by_book_count(data, thresholds=[5, 10, 15, 20]):
    """
    Analyze leaf categories by different book count thresholds
    """
    def find_leaf_categories(data):
        leaves = []
        
        for category_name, category_data in data.items():
            if category_name in ['book_count', 'url', 'subcategories']:
                continue
                
            has_subcategories = False
            if "subcategories" in category_data:
                subcats = category_data["subcategories"]
                if subcats and isinstance(subcats, dict):
                    for key in subcats.keys():
                        if key not in ['book_count', 'url', 'subcategories']:
                            has_subcategories = True
                            leaves.extend(find_leaf_categories(subcats))
                            break
            
            if not has_subcategories:
                leaves.append({
                    'name': category_name,
                    'book_count': category_data.get('book_count', 0)
                })
        
        return leaves
    
    # Get all leaf categories
    all_leaves = find_leaf_categories(data)
    
    print(f"Total leaf categories: {len(all_leaves)}\n")
    
    # Analyze by thresholds
    for threshold in sorted(thresholds):
        categories_below = [cat for cat in all_leaves if cat['book_count'] < threshold]
        print(f"Leaf categories with < {threshold:2d} books: {len(categories_below):3d} ({len(categories_below)/len(all_leaves)*100:.1f}%)")
    
    # Show categories with 0 books
    zero_book_categories = [cat for cat in all_leaves if cat['book_count'] == 0]
    if zero_book_categories:
        print(f"\nCategories with 0 books ({len(zero_book_categories)}):")
        for cat in zero_book_categories[:10]:  # Show first 10
            print(f"  - {cat['name']}")
        if len(zero_book_categories) > 10:
            print(f"  ... and {len(zero_book_categories) - 10} more")

# Usage
with open('category_tree_modified.json', 'r', encoding='utf-8') as file:
    data = json.load(file)

analyze_leaf_categories_by_book_count(data)

Total leaf categories: 799

Leaf categories with <  5 books: 404 (50.6%)
Leaf categories with < 10 books: 541 (67.7%)
Leaf categories with < 15 books: 624 (78.1%)
Leaf categories with < 20 books: 661 (82.7%)

Categories with 0 books (1):
  - تحقیق، تالیف و تصنیف کتاب


In [13]:
import json

def find_leaf_categories_with_few_books(data, max_books=20, current_path=""):
    """
    Find all leaf categories (no subcategories) with book count less than max_books
    """
    leaf_categories = []
    
    for category_name, category_data in data.items():
        # Skip metadata fields
        if category_name in ['book_count', 'url', 'subcategories']:
            continue
            
        # Build current path
        new_path = f"{current_path} > {category_name}" if current_path else category_name
        
        # Check if this category has any actual subcategories
        has_actual_subcategories = False
        
        if "subcategories" in category_data:
            subcats = category_data["subcategories"]
            if subcats and isinstance(subcats, dict):
                # Look for actual category names (not metadata)
                for key in subcats.keys():
                    if key not in ['book_count', 'url', 'subcategories']:
                        has_actual_subcategories = True
                        # Recursively check subcategories
                        leaf_categories.extend(
                            find_leaf_categories_with_few_books(subcats, max_books, new_path)
                        )
                        break
        
        # If no actual subcategories found, this is a leaf category
        if not has_actual_subcategories:
            book_count = category_data.get('book_count', 0)
            if book_count < max_books:
                leaf_categories.append({
                    'name': category_name,
                    'full_path': new_path,
                    'book_count': book_count,
                    'url': category_data.get('url', '')
                })
    
    return leaf_categories

def main():
    # Load your JSON data
    with open('category_tree_modified.json', 'r', encoding='utf-8') as file:
        data = json.load(file)
    
    # Find leaf categories with less than 20 books
    max_books = 20
    leaf_cats_few_books = find_leaf_categories_with_few_books(data, max_books)
    
    # Sort by book count (ascending)
    leaf_cats_few_books.sort(key=lambda x: x['book_count'])
    
    print(f"Found {len(leaf_cats_few_books)} leaf categories with less than {max_books} books:\n")
    
    # Display results
    for i, category in enumerate(leaf_cats_few_books, 1):
        print(f"{i:3d}. {category['name']}")
        print(f"     Full Path: {category['full_path']}")
        print(f"     Books: {category['book_count']}")
        if category['url']:
            print(f"     URL: {category['url']}")
        print()
    
    # Summary statistics
    book_counts = [cat['book_count'] for cat in leaf_cats_few_books]
    if book_counts:
        print(f"\nSummary:")
        print(f"Total leaf categories with < {max_books} books: {len(leaf_cats_few_books)}")
        print(f"Minimum books: {min(book_counts)}")
        print(f"Maximum books: {max(book_counts)}")
        print(f"Average books: {sum(book_counts)/len(book_counts):.1f}")
        
        # Count by book count ranges
        ranges = {f"0": 0, f"1-5": 0, f"6-10": 0, f"11-19": 0}
        for count in book_counts:
            if count == 0:
                ranges["0"] += 1
            elif 1 <= count <= 5:
                ranges["1-5"] += 1
            elif 6 <= count <= 10:
                ranges["6-10"] += 1
            elif 11 <= count <= 19:
                ranges["11-19"] += 1
        
        print(f"\nDistribution:")
        for range_name, count in ranges.items():
            if count > 0:
                print(f"  {range_name} books: {count} categories")

if __name__ == "__main__":
    main()

Found 661 leaf categories with less than 20 books:

  1. تحقیق، تالیف و تصنیف کتاب
     Full Path: 14- علوم دانشگاهی > کتاب و كتابداری > تحقیق، تالیف و تصنیف کتاب
     Books: 0
     URL: https://www.ghbook.ir/index.php?option=com_dbook&task=listcats&cat_id=4410&Itemid=&lang=fa

  2. 4- بررسی مکاتب تفسیری
     Full Path: 01- معارف قرآنی > 3-تفسیر > 4- انواع تفسیر > 4- بررسی مکاتب تفسیری
     Books: 1
     URL: https://www.ghbook.ir/index.php?option=com_dbook&task=listcats&cat_id=3222&Itemid=&lang=fa

  3. 5- ورود احادیث موضوعه و اسرائیلیات در تفاسیر
     Full Path: 01- معارف قرآنی > 3-تفسیر > 5- ورود احادیث موضوعه و اسرائیلیات در تفاسیر
     Books: 1
     URL: https://www.ghbook.ir/index.php?option=com_dbook&task=listcats&cat_id=3218&Itemid=&lang=fa

  4. 11- آیات ناسخ و منسوخ
     Full Path: 01- معارف قرآنی > 4- علوم قرآنی > 11- آیات ناسخ و منسوخ
     Books: 1
     URL: https://www.ghbook.ir/index.php?option=com_dbook&task=listcats&cat_id=3734&Itemid=213&lang=fa

  5. 13- تناسب آبات
  

## delete leaf tags with minimum books

In [10]:
import json

def delete_leaves_by_condition(data, condition_type="equal", book_count=5, current_path=""):
    """
    Delete leaf categories based on book count condition
    condition_type: "equal", "less_than", "greater_than", "less_equal", "greater_equal"
    """
    deleted_categories = []
    keys_to_delete = []
    
    for category_name, category_data in data.items():
        if category_name in ['book_count', 'url', 'subcategories']:
            continue
            
        new_path = f"{current_path} > {category_name}" if current_path else category_name
        
        has_actual_subcategories = False
        
        if "subcategories" in category_data:
            subcats = category_data["subcategories"]
            if subcats and isinstance(subcats, dict):
                for key in subcats.keys():
                    if key not in ['book_count', 'url', 'subcategories']:
                        has_actual_subcategories = True
                        modified_subcats, deleted_from_sub = delete_leaves_by_condition(
                            subcats, condition_type, book_count, new_path
                        )
                        data[category_name]["subcategories"] = modified_subcats
                        deleted_categories.extend(deleted_from_sub)
                        break
        
        if not has_actual_subcategories:
            current_book_count = category_data.get('book_count', 0)
            should_delete = False
            
            if condition_type == "equal" and current_book_count == book_count:
                should_delete = True
            elif condition_type == "less_than" and current_book_count < book_count:
                should_delete = True
            elif condition_type == "greater_than" and current_book_count > book_count:
                should_delete = True
            elif condition_type == "less_equal" and current_book_count <= book_count:
                should_delete = True
            elif condition_type == "greater_equal" and current_book_count >= book_count:
                should_delete = True
            
            if should_delete:
                deleted_categories.append({
                    'name': category_name,
                    'full_path': new_path,
                    'book_count': current_book_count,
                    'url': category_data.get('url', '')
                })
                keys_to_delete.append(category_name)
    
    for key in keys_to_delete:
        del data[key]
    
    return data, deleted_categories

def interactive_main():
    # Load your JSON data
    with open('category_tree_modified.json', 'r', encoding='utf-8') as file:
        data = json.load(file)
    
    print("Choose deletion condition:")
    print("1. Equal to")
    print("2. Less than")
    print("3. Greater than")
    print("4. Less than or equal to")
    print("5. Greater than or equal to")
    
    choice = input("Enter your choice (1-5): ").strip()
    book_count = int(input("Enter book count: "))
    
    condition_map = {
        "1": "equal",
        "2": "less_than",
        "3": "greater_than",
        "4": "less_equal",
        "5": "greater_equal"
    }
    
    condition_type = condition_map.get(choice, "equal")
    
    # Perform deletion
    modified_data, deleted_categories = delete_leaves_by_condition(data, condition_type, book_count)
    
    print(f"\nDeleted {len(deleted_categories)} leaf categories with {condition_type.replace('_', ' ')} {book_count} books:\n")
    
    for i, category in enumerate(deleted_categories, 1):
        print(f"{i:3d}. {category['name']} (Books: {category['book_count']})")
    
    # Save the modified data
    output_filename = f'category_tree_cleaned_{condition_type}_{book_count}.json'
    with open(output_filename, 'w', encoding='utf-8') as file:
        json.dump(modified_data, file, ensure_ascii=False, indent=2)
    
    print(f"\nModified data saved to: {output_filename}")

if __name__ == "__main__":
    interactive_main()

Choose deletion condition:
1. Equal to
2. Less than
3. Greater than
4. Less than or equal to
5. Greater than or equal to


Enter your choice (1-5):  4
Enter book count:  20



Deleted 668 leaf categories with less equal 20 books:

  1. 3- تفسیر سور (Books: 11)
  2. 4- تفسیر و تاویل آیات خاص (Books: 13)
  3. 3- مفسر پژوهی (Books: 6)
  4. 3- تفسیر موضوعی و تطبیقی (Books: 19)
  5. 4- بررسی مکاتب تفسیری (Books: 1)
  6. 5- ورود احادیث موضوعه و اسرائیلیات در تفاسیر (Books: 1)
  7. 1- اعجاز قرآن (Books: 15)
  8. 10- امثال قران (Books: 4)
  9. 11- آیات ناسخ و منسوخ (Books: 1)
 10. 12- تاریخ قرآن (Books: 5)
 11. 13- تناسب آبات (Books: 1)
 12. 14- جامعیت قرآن (Books: 1)
 13. 18- مفردات و غرایب قرآن (لغت قرآن) (Books: 1)
 14. 19- کتابت قرآن (Books: 1)
 15. 2- تحریف ناپذیری قرآن (Books: 11)
 16. 3- اختلاف قرائت (Books: 6)
 17. 4- تجوید و قرائت (Books: 16)
 18. 5- شان نزول (Books: 9)
 19. 6- بطن قرآن (تاویل) (Books: 1)
 20. 7- تدبر در قرآن (Books: 3)
 21. 8- تلاوت و روخوانی (Books: 5)
 22. تاریخ گذاری قرآن (Books: 3)
 23. رمضان ماه نزول قران (Books: 3)
 24. شان نزول آیات (Books: 4)
 25. ائمه علیهم السلام در قرآن (Books: 3)
 26. امام حسین علیه السلام در قرآن (Books: 3)
 

In [11]:
print(f"Leaf tags: {quick_leaf_count('category_tree_cleaned_less_equal_20.json')}")

Leaf tags: 237


# find hierarchies with more than 4 levels

In [5]:
import json

def find_deep_hierarchies(data, max_levels=8, current_level=0, current_path=""):
    """
    Find categories that have hierarchy depth greater than max_levels
    Returns categories that are too deep and their full paths
    """
    deep_categories = []
    
    for category_name, category_data in data.items():
        # Skip metadata fields
        if category_name in ['book_count', 'url', 'subcategories']:
            continue
            
        # Build current path
        new_path = f"{current_path} > {category_name}" if current_path else category_name
        
        # Check if current level exceeds the limit
        if current_level > max_levels:
            deep_categories.append({
                'name': category_name,
                'full_path': new_path,
                'level': current_level,
                'book_count': category_data.get('book_count', 0)
            })
        
        # Recursively check subcategories
        if "subcategories" in category_data:
            subcats = category_data["subcategories"]
            if subcats and isinstance(subcats, dict):
                # Look for actual category names (not metadata)
                for key in subcats.keys():
                    if key not in ['book_count', 'url', 'subcategories']:
                        deep_categories.extend(
                            find_deep_hierarchies(subcats, max_levels, current_level + 1, new_path)
                        )
                        break
    
    return deep_categories

def analyze_hierarchy_depth(data, current_level=0):
    """
    Analyze the maximum depth of hierarchy in the data
    """
    max_depth = current_level
    
    for category_name, category_data in data.items():
        if category_name in ['book_count', 'url', 'subcategories']:
            continue
            
        # Check subcategories for deeper levels
        if "subcategories" in category_data:
            subcats = category_data["subcategories"]
            if subcats and isinstance(subcats, dict):
                for key in subcats.keys():
                    if key not in ['book_count', 'url', 'subcategories']:
                        depth = analyze_hierarchy_depth(subcats, current_level + 1)
                        max_depth = max(max_depth, depth)
                        break
    
    return max_depth

def main():
    # Load your JSON data
    with open('category_tree_modified.json', 'r', encoding='utf-8') as file:
        data = json.load(file)
    
    max_allowed_levels = 8
    
    # First, analyze the maximum depth in the entire structure
    max_depth = analyze_hierarchy_depth(data)
    print(f"Maximum hierarchy depth in the entire structure: {max_depth} levels\n")
    
    # Find categories that exceed the level limit
    deep_categories = find_deep_hierarchies(data, max_allowed_levels)
    
    # Sort by level (descending) to see the deepest ones first
    deep_categories.sort(key=lambda x: x['level'], reverse=True)
    
    print(f"Found {len(deep_categories)} categories with more than {max_allowed_levels} levels of hierarchy:\n")
    
    # Display results
    for i, category in enumerate(deep_categories, 1):
        print(f"{i:3d}. Level {category['level']}: {category['name']}")
        print(f"     Full Path: {category['full_path']}")
        print(f"     Books: {category['book_count']}")
        print()
    
    # Show depth distribution
    if deep_categories:
        levels = [cat['level'] for cat in deep_categories]
        level_counts = {}
        for level in levels:
            level_counts[level] = level_counts.get(level, 0) + 1
        
        print(f"\nDepth Distribution (categories > {max_allowed_levels} levels):")
        for level in sorted(level_counts.keys()):
            print(f"  Level {level}: {level_counts[level]} categories")

def interactive_analysis():
    """
    Interactive version to test different depth thresholds
    """
    # with open('category_tree_modified.json', 'r', encoding='utf-8') as file:
    with open('category_tree_cleaned_less_equal_10.json', 'r', encoding='utf-8') as file:
        data = json.load(file)
    
    max_depth = analyze_hierarchy_depth(data)
    print(f"Maximum hierarchy depth in structure: {max_depth} levels\n")
    
    while True:
        try:
            threshold = int(input(f"Enter depth threshold to analyze (1-{max_depth}): "))
            if threshold < 1 or threshold > max_depth:
                print(f"Please enter a number between 1 and {max_depth}")
                continue
            
            deep_categories = find_deep_hierarchies(data, threshold)
            deep_categories.sort(key=lambda x: x['level'], reverse=True)
            
            print(f"\nCategories with more than {threshold} levels: {len(deep_categories)}")
            
            if deep_categories:
                # Show the 5 deepest categories
                print(f"\nTop 5 deepest categories:")
                for i, category in enumerate(deep_categories[:5], 1):
                    print(f"  {i}. Level {category['level']}: {category['name']}")
                    print(f"     Path: {category['full_path']}")
                
                # Show depth distribution
                levels = [cat['level'] for cat in deep_categories]
                level_counts = {}
                for level in levels:
                    level_counts[level] = level_counts.get(level, 0) + 1
                
                print(f"\nDepth distribution:")
                for level in sorted(level_counts.keys()):
                    print(f"  Level {level}: {level_counts[level]} categories")
            
            print("\n" + "="*50 + "\n")
            
            continue_analysis = input("Analyze another threshold? (y/n): ").lower().strip()
            if continue_analysis != 'y':
                break
                
        except ValueError:
            print("Please enter a valid number")
        except KeyboardInterrupt:
            break

if __name__ == "__main__":
    print("Choose analysis mode:")
    print("1. Single analysis (default: >8 levels)")
    print("2. Interactive analysis (test different thresholds)")
    
    choice = input("Enter your choice (1 or 2): ").strip()
    
    if choice == "2":
        interactive_analysis()
    else:
        main()

Choose analysis mode:
1. Single analysis (default: >8 levels)
2. Interactive analysis (test different thresholds)


Enter your choice (1 or 2):  2


Maximum hierarchy depth in structure: 7 levels



Enter depth threshold to analyze (1-7):  6



Categories with more than 6 levels: 2

Top 5 deepest categories:
  1. Level 7: جنگ ایران و عراق - دفاع مقدس
     Path: 16- تاریخ و  جغرافیا > 2- جهان > آسیا > ایران > ایران بعد از اسلام > تاریخ معاصر ایران > 3- انقلاب اسلامی > جنگ ایران و عراق - دفاع مقدس
  2. Level 7: مقام معظم رهبری آیت الله سید علی خامنه ای
     Path: 16- تاریخ و  جغرافیا > 2- جهان > آسیا > ایران > ایران بعد از اسلام > تاریخ معاصر ایران > 3- انقلاب اسلامی > مقام معظم رهبری آیت الله سید علی خامنه ای

Depth distribution:
  Level 7: 2 categories




Analyze another threshold? (y/n):  y
Enter depth threshold to analyze (1-7):  5



Categories with more than 5 levels: 7

Top 5 deepest categories:
  1. Level 7: جنگ ایران و عراق - دفاع مقدس
     Path: 16- تاریخ و  جغرافیا > 2- جهان > آسیا > ایران > ایران بعد از اسلام > تاریخ معاصر ایران > 3- انقلاب اسلامی > جنگ ایران و عراق - دفاع مقدس
  2. Level 7: مقام معظم رهبری آیت الله سید علی خامنه ای
     Path: 16- تاریخ و  جغرافیا > 2- جهان > آسیا > ایران > ایران بعد از اسلام > تاریخ معاصر ایران > 3- انقلاب اسلامی > مقام معظم رهبری آیت الله سید علی خامنه ای
  3. Level 6: 1- علائم حتمی
     Path: 04- معصومین علیهم السلام > 1- چهارده معصوم علیهم السلام > 14- امام زمان، حضرت مهدی (عجل الله تعالی فرجه) > 13- مهدویت امام زمان، حضرت مهدی (عجل الله تعالی فرجه) > 3- ظهور > علائم ظهور > 1- علائم حتمی
  4. Level 6: 1- مشروطه
     Path: 16- تاریخ و  جغرافیا > 2- جهان > آسیا > ایران > ایران بعد از اسلام > تاریخ معاصر ایران > 1- مشروطه
  5. Level 6: 2- حكومت پهلوی
     Path: 16- تاریخ و  جغرافیا > 2- جهان > آسیا > ایران > ایران بعد از اسلام > تاریخ معاصر ایران > 2- حكومت پهلوی

Depth di

Analyze another threshold? (y/n):  y
Enter depth threshold to analyze (1-7):  4



Categories with more than 4 levels: 13

Top 5 deepest categories:
  1. Level 7: جنگ ایران و عراق - دفاع مقدس
     Path: 16- تاریخ و  جغرافیا > 2- جهان > آسیا > ایران > ایران بعد از اسلام > تاریخ معاصر ایران > 3- انقلاب اسلامی > جنگ ایران و عراق - دفاع مقدس
  2. Level 7: مقام معظم رهبری آیت الله سید علی خامنه ای
     Path: 16- تاریخ و  جغرافیا > 2- جهان > آسیا > ایران > ایران بعد از اسلام > تاریخ معاصر ایران > 3- انقلاب اسلامی > مقام معظم رهبری آیت الله سید علی خامنه ای
  3. Level 6: 1- علائم حتمی
     Path: 04- معصومین علیهم السلام > 1- چهارده معصوم علیهم السلام > 14- امام زمان، حضرت مهدی (عجل الله تعالی فرجه) > 13- مهدویت امام زمان، حضرت مهدی (عجل الله تعالی فرجه) > 3- ظهور > علائم ظهور > 1- علائم حتمی
  4. Level 6: 1- مشروطه
     Path: 16- تاریخ و  جغرافیا > 2- جهان > آسیا > ایران > ایران بعد از اسلام > تاریخ معاصر ایران > 1- مشروطه
  5. Level 6: 2- حكومت پهلوی
     Path: 16- تاریخ و  جغرافیا > 2- جهان > آسیا > ایران > ایران بعد از اسلام > تاریخ معاصر ایران > 2- حكومت پهلوی

Depth d

Analyze another threshold? (y/n):  n


# delete a specific leaf tag

In [5]:
import json

def delete_specific_leaf_by_name(data, target_name, current_path=""):
    """
    Delete a specific leaf node by its exact name
    Returns: (modified_data, deletion_info)
    """
    deleted_info = None
    keys_to_delete = []
    
    for category_name, category_data in data.items():
        # Skip metadata fields
        if category_name in ['book_count', 'url', 'subcategories']:
            continue
            
        # Build current path
        new_path = f"{current_path} > {category_name}" if current_path else category_name
        
        # Check if this category has any actual subcategories
        has_actual_subcategories = False
        
        if "subcategories" in category_data:
            subcats = category_data["subcategories"]
            if subcats and isinstance(subcats, dict):
                # Look for actual category names (not metadata)
                for key in subcats.keys():
                    if key not in ['book_count', 'url', 'subcategories']:
                        has_actual_subcategories = True
                        # Recursively process subcategories
                        modified_subcats, deleted_from_sub = delete_specific_leaf_by_name(
                            subcats, target_name, new_path
                        )
                        data[category_name]["subcategories"] = modified_subcats
                        if deleted_from_sub:
                            deleted_info = deleted_from_sub
                        break
        
        # If no actual subcategories found, this is a leaf category
        if not has_actual_subcategories:
            if category_name == target_name:
                deleted_info = {
                    'name': category_name,
                    'full_path': new_path,
                    'book_count': category_data.get('book_count', 0),
                    'url': category_data.get('url', '')
                }
                keys_to_delete.append(category_name)
    
    # Delete the identified categories
    for key in keys_to_delete:
        del data[key]
    
    return data, deleted_info

def delete_specific_leaf_by_path(data, target_path):
    """
    Delete a specific leaf node by its full path
    """
    path_parts = [part.strip() for part in target_path.split('>')]
    
    def delete_by_path_recursive(data, path_parts, current_index=0):
        if current_index >= len(path_parts):
            return data, None
        
        current_target = path_parts[current_index]
        deleted_info = None
        
        for category_name, category_data in data.items():
            if category_name in ['book_count', 'url', 'subcategories']:
                continue
                
            if category_name == current_target:
                # If this is the last part of the path, delete it
                if current_index == len(path_parts) - 1:
                    # Check if it's actually a leaf
                    has_subcategories = False
                    if "subcategories" in category_data:
                        subcats = category_data["subcategories"]
                        if subcats and isinstance(subcats, dict):
                            for key in subcats.keys():
                                if key not in ['book_count', 'url', 'subcategories']:
                                    has_subcategories = True
                                    break
                    
                    if not has_subcategories:
                        deleted_info = {
                            'name': category_name,
                            'full_path': ' > '.join(path_parts),
                            'book_count': category_data.get('book_count', 0),
                            'url': category_data.get('url', '')
                        }
                        del data[category_name]
                    else:
                        print(f"Warning: '{target_path}' is not a leaf node (it has subcategories)")
                    return data, deleted_info
                else:
                    # Continue deeper in the path
                    if "subcategories" in category_data:
                        modified_subcats, deleted_info = delete_by_path_recursive(
                            category_data["subcategories"], path_parts, current_index + 1
                        )
                        data[category_name]["subcategories"] = modified_subcats
                        return data, deleted_info
        
        return data, None
    
    return delete_by_path_recursive(data, path_parts)

def find_leaf_nodes(data, current_path=""):
    """
    Find all leaf nodes for reference
    """
    leaves = []
    
    for category_name, category_data in data.items():
        if category_name in ['book_count', 'url', 'subcategories']:
            continue
            
        new_path = f"{current_path} > {category_name}" if current_path else category_name
        
        has_subcategories = False
        if "subcategories" in category_data:
            subcats = category_data["subcategories"]
            if subcats and isinstance(subcats, dict):
                for key in subcats.keys():
                    if key not in ['book_count', 'url', 'subcategories']:
                        has_subcategories = True
                        leaves.extend(find_leaf_nodes(subcats, new_path))
                        break
        
        if not has_subcategories:
            leaves.append({
                'name': category_name,
                'full_path': new_path,
                'book_count': category_data.get('book_count', 0)
            })
    
    return leaves

def main():
    # Load your JSON data
    
    # with open('category_tree_modified.json', 'r', encoding='utf-8') as file:
    with open('category_tree_cleaned_less_equal_20.json', 'r', encoding='utf-8') as file:
        data = json.load(file)
    
    # First, show all leaf nodes for reference
    print("Finding all leaf nodes...")
    all_leaves = find_leaf_nodes(data)
    print(f"Total leaf nodes: {len(all_leaves)}\n")
    
    # Show first 20 leaf nodes as examples
    print("First 20 leaf nodes:")
    for i, leaf in enumerate(all_leaves[:20], 1):
        print(f"{i:2d}. {leaf['name']}")
        print(f"    Path: {leaf['full_path']}")
        print(f"    Books: {leaf['book_count']}")
        print()
    
    # Ask user how they want to delete
    print("Choose deletion method:")
    print("1. Delete by exact name")
    print("2. Delete by full path")
    
    choice = input("Enter your choice (1 or 2): ").strip()
    
    if choice == "1":
        target = input("Enter the exact name of the leaf node to delete: ").strip()
        modified_data, deleted_info = delete_specific_leaf_by_name(data, target)
    elif choice == "2":
        target = input("Enter the full path of the leaf node to delete: ").strip()
        modified_data, deleted_info = delete_specific_leaf_by_path(data, target)
    else:
        print("Invalid choice")
        return
    
    if deleted_info:
        print(f"\n✅ Successfully deleted:")
        print(f"Name: {deleted_info['name']}")
        print(f"Path: {deleted_info['full_path']}")
        print(f"Books: {deleted_info['book_count']}")
        
        # Save the modified data
        output_filename = 'category_tree_after_deletion.json'
        with open(output_filename, 'w', encoding='utf-8') as file:
            json.dump(modified_data, file, ensure_ascii=False, indent=2)
        
        print(f"\nModified data saved to: {output_filename}")
    else:
        print(f"\n❌ No leaf node found with the specified {'name' if choice == '1' else 'path'}")

def delete_multiple_leaves():
    """
    Function to delete multiple specific leaf nodes
    """
    # with open('category_tree_modified.json', 'r', encoding='utf-8') as file:
    with open('category_tree_cleaned_less_equal_20.json', 'r', encoding='utf-8') as file:
        data = json.load(file)
    
    # Show leaf nodes first
    all_leaves = find_leaf_nodes(data)
    print("Leaf nodes available:")
    for i, leaf in enumerate(all_leaves[:30], 1):  # Show first 30
        print(f"{i:2d}. {leaf['name']} (Books: {leaf['book_count']})")
    
    print("\nEnter the names of leaf nodes to delete (one per line, empty line to finish):")
    nodes_to_delete = []
    while True:
        node_name = input().strip()
        if not node_name:
            break
        nodes_to_delete.append(node_name)
    
    deleted_count = 0
    for node_name in nodes_to_delete:
        data, deleted_info = delete_specific_leaf_by_name(data, node_name)
        if deleted_info:
            print(f"✅ Deleted: {deleted_info['name']}")
            deleted_count += 1
        else:
            print(f"❌ Not found: {node_name}")
    
    if deleted_count > 0:
        output_filename = 'category_tree_multiple_deletions.json'
        with open(output_filename, 'w', encoding='utf-8') as file:
            json.dump(data, file, ensure_ascii=False, indent=2)
        print(f"\nSaved modified data to: {output_filename}")

if __name__ == "__main__":
    print("Choose mode:")
    print("1. Delete single leaf node")
    print("2. Delete multiple leaf nodes")
    
    mode_choice = input("Enter choice (1 or 2): ").strip()
    
    if mode_choice == "2":
        delete_multiple_leaves()
    else:
        main()

Choose mode:
1. Delete single leaf node
2. Delete multiple leaf nodes


Enter choice (1 or 2):  1


Finding all leaf nodes...
Total leaf nodes: 237

First 20 leaf nodes:
 1. 1- دانستنیها و کلیات و ...
    Path: 01- معارف قرآنی > 1- دانستنیها و کلیات و ...
    Books: 386

 2. 2- قرآن (متن و ترجمه و آوانوشت)
    Path: 01- معارف قرآنی > 2- قرآن (متن و ترجمه و آوانوشت)
    Books: 295

 3. 1- تفسیر اهل بیت علیهم السلام
    Path: 01- معارف قرآنی > 3-تفسیر > 01-کتب تفسیر > 1- تفسیر اهل بیت علیهم السلام
    Books: 47

 4. 2- سایر کتب تفسیر
    Path: 01- معارف قرآنی > 3-تفسیر > 01-کتب تفسیر > 2- سایر کتب تفسیر
    Books: 455

 5. 2- تفسیر پژوهی
    Path: 01- معارف قرآنی > 3-تفسیر > 2- تفسیر پژوهی
    Books: 34

 6. 4- انواع تفسیر
    Path: 01- معارف قرآنی > 3-تفسیر > 4- انواع تفسیر
    Books: 24

 7. 20- علم مفاهیم شناسی در قرآن
    Path: 01- معارف قرآنی > 4- علوم قرآنی > 20- علم مفاهیم شناسی در قرآن
    Books: 147

 8. نزول قرآن
    Path: 01- معارف قرآنی > 4- علوم قرآنی > نزول قرآن
    Books: 11

 9. اهل بیت علیهم السلام در قرآن
    Path: 01- معارف قرآنی > 5- موضوعات قرآنی > اهل بیت علیهم ال

Enter your choice (1 or 2):  1
Enter the exact name of the leaf node to delete:  1- علائم حتمی



✅ Successfully deleted:
Name: 1- علائم حتمی
Path: 04- معصومین علیهم السلام > 1- چهارده معصوم علیهم السلام > 14- امام زمان، حضرت مهدی (عجل الله تعالی فرجه) > 13- مهدویت امام زمان، حضرت مهدی (عجل الله تعالی فرجه) > 3- ظهور > علائم ظهور > 1- علائم حتمی
Books: 3

Modified data saved to: category_tree_after_deletion.json


In [6]:
print(f"Leaf tags: {quick_leaf_count('category_tree_after_deletion.json')}")

Leaf tags: 237


# create a new hierarchical category

In [5]:
import json

# Define your new hierarchical structure without book_count
new_categorization = {
    "01- معارف اسلامی": {
        "subcategories": {
            "معارف قرآنی": {
                "subcategories": {
                    "علوم و موضوعات قرآنی": {},
                    "تفسیر قرآن": {},
                    "قرآن (متن، ترجمه و آوانوشت)": {}
                }
            },
            "عقاید اسلامی": {
                "subcategories": {
                    "اصول دین": {
                        "subcategories": {
                            "توحید": {},
                            "نبوت": {},
                            "معاد": {},
                            "امامت": {
                                "subcategories": {
                                    "امامت در اسلام": {},
                                    "مهدویت و ظهور": {}
                                }
                            }
                        }
                    }
                }
            },
            "معارف نهج البلاغه": {},
            "معارف صحیفه سجادیه": {},
            "حدیث و روایات": {},
            "اعتقادات و جهان‌بینی اسلامی": {}
        }
    },
    "02- شخصیت‌ها": {
        "subcategories": {
            "شخصیت‌های علمی، سیاسی و اجتماعی": {},
            "معصومین": {
                "subcategories": {
                    "چهارده معصوم (ع)": {
                        "subcategories": {
                            "پیامبر اکرم، حضرت محمد (ص)": {},
                            "امام علی، امیر المومنین (ع)": {},
                            "حضرت فاطمه زهرا (سلام الله علیها)": {},
                            "امام حسن مجتبی (علیه السلام)": {},
                            "امام حسین، سید الشهداء (علیه السلام)": {},
                            "امام زین العابدین سجاد (علیه السلام)": {},
                            "امام محمد باقر (علیه السلام)": {},
                            "امام جعفر صادق (علیه السلام)": {},
                            "امام كاظم، موسی بن جعفر (علیه السلام)": {},
                            "امام رضا، علی بن موسی (علیه السلام)": {},
                            "امام محمد تقی جواد (علیه السلام)": {},
                            "امام علی نقی هادی (علیه السلام)": {},
                            "امام حسن عسكری (علیه السلام)": {},
                            "امام زمان، حضرت مهدی (عجل الله تعالی فرجه)": {}
                        }
                    },
                    "پیامبران": {}
                }
            }
        }
    },
    "03- علوم حوزوی": {
        "subcategories": {
            "معارف": {
                "subcategories": {
                    "فلسفه و منطق": {},
                    "کلام": {},
                    "الهیات و عرفان": {}
                }
            },
            "فقه و احکام": {
                "subcategories": {
                    "اصول فقه": {},
                    "احکام شرعی": {}
                }
            },
            "رجال و درایه": {},
            "متون درسی حوزوی (پایه تا خارج)": {}
        }
    },
    "04- علوم غیرحوزوی": {
        "subcategories": {
            "پزشکی و طب سنتی": {},
            "زبان و ادبیات": {},
            "روان‌شناسی": {},
            "اقتصاد": {},
            "حرفه و هنر": {},
            "حقوق": {},
            "مدیریت": {},
            "کتاب و کتابداری": {},
            "علوم و دانش عمومی": {},
            "جامعه‌شناسی و علوم اجتماعی": {},
            "علوم غریبه و خرق عادت": {},
            "علوم سیاسی": {
                "subcategories": {
                    "جنگ و دفاع": {},
                    "حکومت و سیاست": {}
                }
            },
            "فناوری": {
                "subcategories": {
                    "رسانه‌ها و ارتباطات": {},
                    "تکنولوژی‌های نوین": {}
                }
            }
        }
    },
    "05- اخلاق و تربیت": {
        "subcategories": {
            "فضائل اخلاقی": {
                "subcategories": {
                    "شهادت و آزادگی": {},
                    "تقوا و تهذیب نفس": {},
                    "ارزش‌های اخلاقی": {}
                }
            },
            "اخلاق اجتماعی": {
                "subcategories": {
                    "امر به معروف و نهی از منکر": {},
                    "حجاب و عفاف": {},
                    "آداب معاشرت و رفتار اجتماعی": {}
                }
            },
            "خانواده": {
                "subcategories": {
                    "مهارت‌های زندگی و اصول تربیت": {},
                    "ازدواج": {}
                }
            }
        }
    },
    "06- ادیان و مذاهب": {
        "subcategories": {
            "ادیان ابراهیمی": {
                "subcategories": {
                    "اسلام": {
                        "subcategories": {
                            "تشیع": {},
                            "تسنن": {},
                            "شیعه و سنی": {}
                        }
                    },
                    "مسیحیت": {},
                    "یهودیت": {}
                }
            },
            "سایر ادیان، فرق و مکاتب": {
                "subcategories": {
                    "بابیت و بهائیت": {},
                    "وهابیت": {},
                    "فرق و جریان‌های فکری": {}
                }
            },
            "مطالعات تطبیقی و گفتگوی ادیان (دین و اندیشه)": {}
        }
    },
    "07- مناسبت‌ها": {
        "subcategories": {
            "اعیاد": {
                "subcategories": {
                    "غدیر خم": {},
                    "سایر اعیاد اسلامی": {}
                }
            },
            "عزاداری‌ها و سوگواری": {},
            "مناسبت‌های ویژه": {}
        }
    },
    "08- اعمال عبادی": {
        "subcategories": {
            "نماز": {},
            "روزه": {},
            "حج و عمره": {},
            "ادعیه و زیارات": {},
            "سایر اعمال عبادی": {}
        }
    },
    "09- تاریخ": {
        "subcategories": {
            "تاریخ اسلام و عرب": {
                "subcategories": {
                    "تاریخ صدر اسلام": {},
                    "تاریخ ائمه": {},
                    "تاریخ خلفا": {}
                }
            },
            "تاریخ ایران": {
                "subcategories": {
                    "تاریخ معاصر ایران": {},
                    "تاریخ تمدن ایران": {}
                }
            },
            "تاریخ جهان": {
                "subcategories": {
                    "تاریخ معاصر جهان": {},
                    "تاریخ ملل و تمدن‌ها": {}
                }
            },
            "اماکن مقدسه": {}
        }
    }
}

# Save to new JSON file
with open('new_categorization.json', 'w', encoding='utf-8') as f:
    json.dump(new_categorization, f, ensure_ascii=False, indent=2)

print("✅ New categorization saved to 'new_categorization.json'")

✅ New categorization saved to 'new_categorization.json'


# second version (consiced)

In [20]:
import json

# Define your new hierarchical structure without book_count
new_categorization = {
    "01- معارف اسلامی": {
        "subcategories": {
            "معارف قرآنی": {
                "subcategories": {
                    "علوم و موضوعات قرآنی": {},
                    "تفسیر قرآن": {},
                    "قرآن (متن، ترجمه و آوانوشت)": {}
                }
            },
            "عقاید اسلامی": {
                "subcategories": {
                    "اصول دین": {
                        "subcategories": {
                            "توحید": {},
                            "نبوت": {},
                            "معاد": {},
                            "امامت": {
                                "subcategories": {
                                    "امامت در اسلام": {},
                                    "مهدویت و ظهور": {}
                                }
                            }
                        }
                    }
                }
            },
            "معارف نهج البلاغه": {},
            "معارف صحیفه سجادیه": {},
            "حدیث و روایات": {},
            "اعتقادات و جهان‌بینی اسلامی": {}
        }
    },
    "02- شخصیت‌ها": {
        "subcategories": {
            "شخصیت‌های علمی، سیاسی و اجتماعی": {},
            "معصومین": {
                "subcategories": {
                    "چهارده معصوم (ع)": {
                        "subcategories": {
                            "پیامبر اکرم، حضرت محمد (ص)": {},
                            "امام علی، امیر المومنین (ع)": {},
                            "حضرت فاطمه زهرا (سلام الله علیها)": {},
                            "امام حسین، سید الشهداء (علیه السلام)": {},
                            "امام زمان، حضرت مهدی (عجل الله تعالی فرجه)": {},
                            "سایر ائمه اطهار":{}
                        }
                    },
                    "پیامبران": {}
                }
            }
        }
    },
    "03- علوم حوزوی": {
        "subcategories": {
            "معارف": {
                "subcategories": {
                    "فلسفه و منطق": {},
                    "کلام": {},
                    "الهیات و عرفان": {}
                }
            },
            "فقه و احکام": {
                "subcategories": {
                    "اصول فقه": {},
                    "احکام شرعی": {}
                }
            },
            "رجال و درایه": {},
            "متون درسی حوزوی (پایه تا خارج)": {}
        }
    },
    "04- علوم غیرحوزوی": {
        "subcategories": {
            "پزشکی و طب سنتی": {},
            "زبان و ادبیات": {},
            "روان‌شناسی": {},
            "اقتصاد": {},
            "حقوق": {},
            "مدیریت": {},
            "کتاب و کتابداری": {},
            "علوم و دانش عمومی": {},
            "جامعه‌شناسی و علوم اجتماعی": {},
            "علوم سیاسی": {
                "subcategories": {
                    "جنگ و دفاع": {},
                    "حکومت و سیاست": {}
                }
            },
            "فناوری": {
                "subcategories": {
                    "رسانه‌ها و ارتباطات": {},
                    "تکنولوژی‌های نوین": {}
                }
            }
        }
    },
    "05- اخلاق و تربیت": {
        "subcategories": {
            "فضائل اخلاقی": {
                "subcategories": {
                    "شهادت و آزادگی": {},
                    "تقوا و تهذیب نفس": {},
                    "ارزش‌های اخلاقی": {}
                }
            },
            "اخلاق اجتماعی": {
                "subcategories": {
                    "امر به معروف و نهی از منکر": {},
                    "حجاب و عفاف": {},
                    "آداب معاشرت و رفتار اجتماعی": {}
                }
            },
            "خانواده": {
                "subcategories": {
                    "مهارت‌های زندگی و اصول تربیت": {},
                    "ازدواج": {}
                }
            }
        }
    },
    "06- ادیان و مذاهب": {
        "subcategories": {
            "ادیان ابراهیمی": {
                "subcategories": {
                    "اسلام": {
                        "subcategories": {
                            "تشیع": {},
                            "تسنن": {},
                            "شیعه و سنی": {}
                        }
                    },
                    "مسیحیت": {},
                    "یهودیت": {}
                }
            },
            "سایر ادیان، فرق و مکاتب": {
                "subcategories": {
                    "بابیت و بهائیت": {},
                    "وهابیت": {},
                    "فرق و جریان‌های فکری": {}
                }
            },
            "مطالعات تطبیقی و گفتگوی ادیان (دین و اندیشه)": {}
        }
    },

    "07- تاریخ": {
        "subcategories": {
            "تاریخ اسلام و عرب": {
                "subcategories": {
                    "تاریخ صدر اسلام": {},
                    "تاریخ ائمه": {},
                    "تاریخ خلفا": {}
                }
            },
            "تاریخ ایران": {
                "subcategories": {
                    "تاریخ معاصر ایران": {},
                    "تاریخ تمدن ایران": {}
                }
            },
            "تاریخ جهان": {
                "subcategories": {
                    "تاریخ معاصر جهان": {},
                    "تاریخ ملل و تمدن‌ها": {}
                }
            },
            "اماکن مقدسه": {},
            "مناسبت‌ها (اعیاد، سوگواری و ...)": {}
        }
    }
}

# Save to new JSON file
with open('new_categorization.json', 'w', encoding='utf-8') as f:
    json.dump(new_categorization, f, ensure_ascii=False, indent=2)

print("✅ New categorization saved to 'new_categorization.json'")

✅ New categorization saved to 'new_categorization.json'


In [15]:
import json

def count_leaf_tags(data):
    """
    Count all leaf tags in the categorization structure
    """
    leaf_count = 0
    
    def count_leaves_recursive(node):
        nonlocal leaf_count
        
        # If this node has subcategories, recursively check them
        if "subcategories" in node and node["subcategories"]:
            for subcategory_name, subcategory_data in node["subcategories"].items():
                count_leaves_recursive(subcategory_data)
        else:
            # This is a leaf node
            leaf_count += 1
    
    # Start counting from the main categories
    for main_category, category_data in new_categorization.items():
        count_leaves_recursive(category_data)
    
    return leaf_count

# Count leaf tags
leaf_tags_count = count_leaf_tags(new_categorization)
print(f"Total leaf tags: {leaf_tags_count}")

Total leaf tags: 66


In [16]:
import json

def count_leaf_tags(data):
    """
    Count all leaf tags in the categorization structure
    """
    leaf_count = 0
    leaf_tags = []
    
    def count_leaves_recursive(node, path=""):
        nonlocal leaf_count
        
        # If this node has subcategories, recursively check them
        if "subcategories" in node and node["subcategories"]:
            for subcategory_name, subcategory_data in node["subcategories"].items():
                new_path = f"{path} > {subcategory_name}" if path else subcategory_name
                count_leaves_recursive(subcategory_data, new_path)
        else:
            # This is a leaf node
            leaf_count += 1
            leaf_tags.append(path)
    
    # Start counting from the main categories
    for main_category, category_data in data.items():
        count_leaves_recursive(category_data, main_category)
    
    return leaf_count, leaf_tags

# Count leaf tags
leaf_count, all_leaves = count_leaf_tags(new_categorization)

print(f"Total leaf tags: {leaf_count}\n")
print("All leaf tags:")
for i, tag in enumerate(all_leaves, 1):
    print(f"{i:3d}. {tag}")

Total leaf tags: 66

All leaf tags:
  1. 01- معارف اسلامی > معارف قرآنی > علوم و موضوعات قرآنی
  2. 01- معارف اسلامی > معارف قرآنی > تفسیر قرآن
  3. 01- معارف اسلامی > معارف قرآنی > قرآن (متن، ترجمه و آوانوشت)
  4. 01- معارف اسلامی > عقاید اسلامی > اصول دین > توحید
  5. 01- معارف اسلامی > عقاید اسلامی > اصول دین > نبوت
  6. 01- معارف اسلامی > عقاید اسلامی > اصول دین > معاد
  7. 01- معارف اسلامی > عقاید اسلامی > اصول دین > امامت > امامت در اسلام
  8. 01- معارف اسلامی > عقاید اسلامی > اصول دین > امامت > مهدویت و ظهور
  9. 01- معارف اسلامی > معارف نهج البلاغه
 10. 01- معارف اسلامی > معارف صحیفه سجادیه
 11. 01- معارف اسلامی > حدیث و روایات
 12. 01- معارف اسلامی > اعتقادات و جهان‌بینی اسلامی
 13. 02- شخصیت‌ها > شخصیت‌های علمی، سیاسی و اجتماعی
 14. 02- شخصیت‌ها > معصومین > چهارده معصوم (ع) > پیامبر اکرم، حضرت محمد (ص)
 15. 02- شخصیت‌ها > معصومین > چهارده معصوم (ع) > امام علی، امیر المومنین (ع)
 16. 02- شخصیت‌ها > معصومین > چهارده معصوم (ع) > حضرت فاطمه زهرا (سلام الله علیها)
 17. 02- شخصیت‌ه

In [19]:
def generate_text_hierarchy():
    with open('new_categorization.json', 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    def print_hierarchy(node, indent=0, is_last=True, prefix=""):
        for i, (key, value) in enumerate(node.items()):
            if key == 'subcategories':
                continue
            
            # Determine the tree branch characters
            if indent == 0:
                branch = "📁 "
            elif is_last and i == len(node) - 1:
                branch = "└── "
            else:
                branch = "├── "
            
            # Remove numbering for cleaner display
            display_name = key.replace(/^\d+-\\s*/, "")
            
            print(" " * indent + branch + display_name)
            
            if "subcategories" in value and value["subcategories"]:
                new_indent = indent + 4
                print_hierarchy(value["subcategories"], new_indent, i == len(node) - 1)
    
    print("🌳 ساختار درختی دسته‌بندی کتابخانه")
    print("=" * 50)
    print_hierarchy(data)
    print("\\n📊 آمار:")
    print(f"   - دسته‌بندی‌های اصلی: {len(data)}")
    
    leaf_count = count_leaf_tags_corrected(data)[0]
    print(f"   - برچسب‌های نهایی: {leaf_count}")

generate_text_hierarchy()

SyntaxError: invalid syntax (227966983.py, line 19)